<a href="https://colab.research.google.com/github/mcristinasanchez-ai/TFG_MATEM-TICAS_M.CristinaS-nchezTim-n/blob/main/PSO_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

# 1. Funciones objetivo que vamos a minimizar
def sphere(x):
    return np.sum(x**2)

def schwefel_1(x):
    return np.sum(np.abs(x)) + np.prod(np.abs(x))

def schwefel_2(x):
    return np.sum([np.sum(x[:i+1])**2 for i in range(len(x))])

def schwefel_3(x):
    return np.max(np.abs(x))

def rosenbrock(x):
    return np.sum(100.0 * (x[1:] - x[:-1]**2.0)**2.0 + (x[:-1] - 1.0)**2.0)

def step(x):
    return np.sum(np.floor(x + 0.5)**2)

def quartic(x):
    return np.sum(np.arange(1, len(x) + 1) * (x**4)) + np.random.random()

def schwefel_4(x):
    return np.sum(-x * np.sin(np.sqrt(np.abs(x))))

def rastrigin(x):
    return np.sum(x**2 - 10 * np.cos(2 * np.pi * x) + 10)

def ackley(x):
    n = float(len(x))
    return -20 * np.exp(-0.2 * np.sqrt(np.sum(x**2) / n)) - np.exp(np.sum(np.cos(2 * np.pi * x)) / n) + 20 + np.exp(1)


funciones_prueba = {
    "Sphere": (sphere, -100, 100),
    "Schwefel 1": (schwefel_1, -10, 10),
    "Schwefel 2": (schwefel_2, -100, 100),
    "Schwefel 3": (schwefel_3, -100, 100),
    "Rosenbrock": (rosenbrock, -30, 30),
    "Step": (step, -100, 100),
    "Quartic": (quartic, -1.28, 1.28),
    "Schwefel 4": (schwefel_4, -500, 500),
    "Rastrigin": (rastrigin, -5.12, 5.12),
    "Ackley": (ackley, -32, 32)
}

# 2. Parámetros
num_particulas = 100 #cantidad de partículas en el espacio de búsqueda
num_iteraciones = 10000 #número pasos máximos que dara el enjambre de partículas en cada intento
dimension = 30 #espacio de búsqueda de 30 dimensiones
w = 0.729 #coeficiente de inercia que controla la tendencia de la partícula a seguir es su dirección actual
c1 = 1.494 #coeficiente cognitivo: cuanto confía la partícula en su propia experiencia
c2 = 1.494 #coeficiente social: cuanto influye el grupo en la partícula

num_ejecuciones = 10 #Numero de ejecuciones por cada función

#Se Almacenan los mejores resultados de cada ejecución
resultados_totales = {}

print("Iniciando optimizaciones...")

for nombre_funcion, (f, limite_inf, limite_sup) in funciones_prueba.items():
    print(f"\nEjecutando 10 veces para: {nombre_funcion}...")
    resultados_funcion = []  # Almacenenamos los 10 mejores resultados globales de esta función

    for ejecucion in range(num_ejecuciones):
        # 3. Inicialización
        posicion = np.random.uniform(limite_inf, limite_sup, (num_particulas, dimension)) #matriz donde se colocan las 100 partículas de forma aleatoria
        velocidad = np.zeros((num_particulas, dimension)) #se inicializa la velocidad a 0, las partículas inicialmente estan paradas

        pmejor_posicion = posicion.copy() #se guarda la mejor posición de la partícula
        pmejor_valor = np.array([f(p) for p in posicion]) #se calcula su valor
        gmejor_posicion = pmejor_posicion[np.argmin(pmejor_valor)].copy() #se guarda la mejor posción del enjambre
        gmejor_valor = np.min(pmejor_valor) #se calcula su valor

        # 4. Bucle principal del algoritmo PSO
        for _ in range(num_iteraciones):
            #Se generan r1 y r2 aleatorios para posteriormente actualizar la velocidad
            #y la posición
            r1 = np.random.rand(num_particulas, dimension)
            r2 = np.random.rand(num_particulas, dimension)

            velocidad = (w * velocidad +
                         c1 * r1 * (pmejor_posicion - posicion) +
                         c2 * r2 * (gmejor_posicion - posicion))

            posicion += velocidad
            #Se usa np.clip para que las partículas no busquen en zonas no válidas
            #si se superan los límites permitidos
            posicion = np.clip(posicion, limite_inf, limite_sup)

            #Se evalúa la nueva posicón comoparando con la propia de la partícula
            #y con la del enjambre
            for i in range(num_particulas):
                valor_actual = f(posicion[i])
                if valor_actual < pmejor_valor[i]:
                    pmejor_valor[i] = valor_actual
                    pmejor_posicion[i] = posicion[i].copy()

                    if valor_actual < gmejor_valor:
                        gmejor_valor = valor_actual
                        gmejor_posicion = posicion[i].copy()

        resultados_funcion.append(gmejor_valor)
        print(f"  Ejecución {ejecucion + 1}: {gmejor_valor:.4e}")

    resultados_totales[nombre_funcion] = resultados_funcion


#CREAMOS LA TABLA CON LOS RESULTADOS DE CADA FUNCIÓN EN LAS 10 Iteraciones y
#Y CALCULAMOS LA MEDIA, MEDIANA Y VARIANZA


# 1. Se crea el Dataframe donde cada columna es una de las funciones de prueba y
#y cada fila son las ejecuiones
df_inicial = pd.DataFrame(resultados_totales)
df_inicial.index = [f"EJECUCIÓN {i+1}" for i in range(num_ejecuciones)]
df_final = df_inicial.copy()

# 2. Se calcula la media, mediana y varianza
df_final.loc['MEDIA'] = df_inicial.mean(axis=0)
df_final.loc['MEDIANA'] = df_inicial.median(axis=0)
df_final.loc['VARIANZA'] = df_inicial.var(axis=0)

# 3. Se muestan los resultados por pantalla y se guardan los resultados en la
#hoja de cáculo
print("\n" + "="*50)
print("TABLA COMPARATIVA FINAL")
print("="*50)
pd.set_option('display.float_format', lambda x: f'{x:.4e}')
print(df_final)
df_final.to_excel("Tabla comparativaPSO1.xlsx")

Iniciando optimizaciones...

Ejecutando 10 veces para: Sphere...
  Ejecución 1: 8.6944e-263
  Ejecución 2: 1.7411e-260
  Ejecución 3: 1.1481e-260
  Ejecución 4: 1.2593e-259
  Ejecución 5: 2.6863e-258
  Ejecución 6: 2.7569e-261
  Ejecución 7: 2.2503e-260
  Ejecución 8: 1.0077e-259
  Ejecución 9: 1.1642e-260
  Ejecución 10: 7.7237e-262

Ejecutando 10 veces para: Schwefel 1...
  Ejecución 1: 2.0000e+01
  Ejecución 2: 7.3142e-124
  Ejecución 3: 3.0000e+01
  Ejecución 4: 2.0000e+01
  Ejecución 5: 2.0000e+01
  Ejecución 6: 1.0000e+01
  Ejecución 7: 3.0000e+01
  Ejecución 8: 1.0000e+01
  Ejecución 9: 1.7729e-120
  Ejecución 10: 2.0000e+01

Ejecutando 10 veces para: Schwefel 2...
  Ejecución 1: 6.6667e+03
  Ejecución 2: 1.0000e+04
  Ejecución 3: 1.5000e+04
  Ejecución 4: 5.7462e-34
  Ejecución 5: 5.0000e+03
  Ejecución 6: 1.2246e-34
  Ejecución 7: 1.5000e+04
  Ejecución 8: 1.4066e-32
  Ejecución 9: 6.6667e+03
  Ejecución 10: 5.0000e+03

Ejecutando 10 veces para: Schwefel 3...
  Ejecución 1: 2.

In [ ]:
import numpy as np

#1.Funciónes objetivo que vamos a minimizar:
def sphere(x):
   return np.sum(x**2)

def schwefel_1(x):
   return np.sum(np.abs(x)) + np.prod(np.abs(x))

def schwefel_2(x):
   return np.sum([np.sum(x[:i+1])**2 for i in range(len(x))])

def schwefel_3(x):
   return np.max(np.abs(x))

def rosenbrock(x):
   return np.sum(100.0 * (x[1:] - x[:-1]**2.0)**2.0 + (x[:-1] - 1.0)**2.0)

def step(x):
   return np.sum(np.floor(x + 0.5)**2)

def quartic(x):
   return np.sum(np.arange(1, len(x) + 1) * (x**4)) + np.random.random()

def schwefel_4(x):
   return np.sum(-x * np.sin(np.sqrt(np.abs(x))))

def rastrigin(x):
   return np.sum(x**2 - 10 * np.cos(2 * np.pi * x) + 10)

def ackley(x):
   n = float(len(x))
   return -20 * np.exp(-0.2 * np.sqrt(np.sum(x**2) / n)) - np.exp(np.sum(np.cos(2 * np.pi * x)) / n) + 20 + np.exp(1)




funciones_prueba = {
    "Sphere": (sphere, -100, 100),
    "Schwefel 1": (schwefel_1, -10, 10),
    "Schwefel 2": (schwefel_2, -100, 100),
    "Schwefel 3": (schwefel_3, -100, 100),
    "Rosenbrock": (rosenbrock, -30, 30),
    "Step": (step, -100, 100),
    "Quartic": (quartic, -1.28, 1.28),
    "Schwefel 4": (schwefel_4, -500, 500),
    "Rastrigin": (rastrigin, -5.12, 5.12),
    "Ackley": (ackley, -32, 32)
}

# 2. Parámetros
num_particulas = 100 #cantidad de partículas del espacio de búsqueda
num_iteraciones = 10000
dimension = 30
w = 0.729   #Inercia que representa la tendencia de la partícula a continuar en su dirección actual
c1 = 1.494 # Coeficiente cognitivo,Pondera la influencia del mejor rendimiento individual
c2 = 1.494 # Coeficiente social, Pondera la influencia del mejor rendimiento global
           #Cuando combinamos ambas con r1 y r2 determinamos si semueven hacía su experiencia pasada c1 o hacía el líder c2
for nombre_funcion, (f, limite_inf, limite_sup) in funciones_prueba.items():
    print(f"\n--- Optimizando: {nombre_funcion} ---")

    # 3. Inicialización
    posicion = np.random.uniform(limite_inf, limite_sup, (num_particulas, dimension))
    velocidad = np.zeros((num_particulas, dimension))

    pmejor_posicion = posicion.copy() #Guarda la mejor posición de cada partícula hasta el momento
    pmejor_valor = np.array([f(p) for p in posicion])
    gmejor_posicion = pmejor_posicion[np.argmin(pmejor_valor)].copy()
    gmejor_valor = np.min(pmejor_valor)

    # 4. PSO loop
    for _ in range(num_iteraciones):
        # Generamos r1 y r2 para el enjambre
        r1 = np.random.rand(num_particulas, dimension)
        r2 = np.random.rand(num_particulas, dimension)
        # Actualización de velocidad
        velocidad = (w * velocidad +
                     c1 * r1 * (pmejor_posicion - posicion) +
                     c2 * r2 * (gmejor_posicion - posicion))


        posicion += velocidad   # Actualización de posición

        # Control de límites (Crucial)
        posicion = np.clip(posicion, limite_inf, limite_sup)

        # Evaluación y actualización de mejores
        for i in range(num_particulas):
            valor_actual = f(posicion[i])
            if valor_actual < pmejor_valor[i]:
                pmejor_valor[i] = valor_actual
                pmejor_posicion[i] = posicion[i].copy()

                # ¿Es mejor que el global?
                # Si la partícula i encuentra un lugar donde el resultado de la función (valor_actual) es menor que el récord global (gmejor_valor):
                # Se destruye el record anterior, gmejor_valor adquiere el nuvo número mas bajo
                # gmejor_posicion guarda una copia exacta de las coordenadas
                if valor_actual < gmejor_valor:
                    gmejor_valor = valor_actual
                    gmejor_posicion = posicion[i].copy()

    print("Optimización terminada")
    #print("Mejor posición encontrada:", gmejor_posicion)
    print("Valor mínimo:", f(gmejor_posicion))